In [ ]:
!pip install langchain langchain-core langchain-community langchain-huggingface chromadb sentence-transformers langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [ ]:
!pip install langchain langchain-core langchain-community langchain-huggingface chromadb sentence-transformers

In [ ]:
!pip install pdfplumber

import pdfplumber
import re
import pandas as pd # 引入 pandas 方便我們用表格來檢視分塊成果

def process_insurance_pdf(file_path, company_name):
    full_text = ""
    print(f"開始解析 {company_name} 旅平險保單...")

    # 2. 讀取 PDF 並萃取文字
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
    except FileNotFoundError:
        return f"錯誤：找不到 {file_path}，請確認是否已上傳至 Colab！"

    # 3. 資料清洗 (Data Cleaning)
    if company_name == "國泰":
        # 濾除國泰保單特有的巨大「樣張」浮水印干擾文字
        full_text = full_text.replace("樣張", "")

    # 去除多餘的空白行，保持段落連貫
    full_text = re.sub(r'\n+', '\n', full_text)

    # 4. 智能語意分塊 (Semantic Chunking)
    # 使用正規表達式匹配「第一條」、「第二十條」等作為切塊錨點
    chunks = re.split(r'(第[一二三四五六七八九十百]+條\s+.*?\n)', full_text)

    processed_chunks = []
    # chunks 的第 0 個元素通常是保單開頭的說明，我們從第 1 個元素開始抓取條款
    for i in range(1, len(chunks), 2):
        clause_title = chunks[i].strip()
        # 確保不會發生 index out of range
        clause_content = chunks[i+1].strip() if (i+1) < len(chunks) else ""

        chunk_data = {
            "保險公司": company_name,
            "條款標題": clause_title,
            # 將公司、標題與內文組合成完整區塊給 AI 讀取
            "完整文本 (Chunk)": f"[{company_name}] {clause_title}\n{clause_content}"
        }
        processed_chunks.append(chunk_data)

    return processed_chunks

# 5. 執行處理
nanshan_chunks = process_insurance_pdf("南山產物旅平險保單條款.pdf", "南山")
cathay_chunks = process_insurance_pdf("國泰旅平險保單條款.pdf", "國泰")

# 6. 將結果轉換為 DataFrame 方便檢視與後續處理
if isinstance(nanshan_chunks, list) and isinstance(cathay_chunks, list):
    all_chunks = nanshan_chunks + cathay_chunks
    df_chunks = pd.DataFrame(all_chunks)

    print(f"\n解析完成！共切出 {len(all_chunks)} 個條款區塊。")
    # 顯示前 5 筆分塊結果讓你看一下效果
    display(df_chunks.head())
else:
    print(nanshan_chunks) # 印出錯誤訊息

In [ ]:
# 載入套件 (修正了 Document 的載入路徑)
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("套件載入完成，準備建立資料庫...")

# 將第一步切好的 chunks 轉換為 LangChain 的 Document 格式
documents = []
for chunk in all_chunks: # 這裡會接續使用你第一步跑出來的 all_chunks
    # 保留 metadata，這對於未來追蹤答案來源 (Citations) 非常重要
    doc = Document(
        page_content=chunk["完整文本 (Chunk)"],
        metadata={
            "保險公司": chunk["保險公司"],
            "條款標題": chunk["條款標題"]
        }
    )
    documents.append(doc)

# 初始化 Embedding 模型
# 選用支援中文的開源模型 shibing624/text2vec-base-chinese
print("正在下載並載入語意向量 (Embedding) 模型，這可能需要一兩分鐘...")
embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")

# 建立 Chroma 向量資料庫並存入資料
print("正在將保險條款轉換為數學向量，並寫入 ChromaDB 資料庫...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./insurance_chroma_db" # 將資料庫存在 Colab 的暫存資料夾中
)

print(f"\n🎉 成功建立向量資料庫！共存入 {vector_db._collection.count()} 筆條款區塊。")

# 實地測試檢索演算法 (Retrieval)
# 測試你的系統是否能捕捉到語意
test_query = "如果我的飛機延誤了五個小時，有理賠嗎？"
print(f"\n🔍 測試檢索問題: '{test_query}'")

# 請資料庫根據語意向量找出最相關的 3 個條款 (k=8)
results = vector_db.similarity_search(test_query, k=8)

for i, res in enumerate(results):
    print(f"\n--- 相關條款 {i+1} ---")
    print(f"來源: {res.metadata['保險公司']} - {res.metadata['條款標題']}")
    # 印出前 150 個字預覽
    print(f"內容: {res.page_content[:150]}...")

套件載入完成，準備建立資料庫...
正在下載並載入語意向量 (Embedding) 模型，這可能需要一兩分鐘...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


正在將保險條款轉換為數學向量，並寫入 ChromaDB 資料庫...

🎉 成功建立向量資料庫！共存入 168 筆條款區塊。

🔍 測試檢索問題: '如果我的飛機延誤了五個小時，有理賠嗎？'

--- 相關條款 1 ---
來源: 南山 - 第二十一條 承保範圍 第二十五條 特別不保事項
內容: [南山] 第二十一條 承保範圍 第二十五條 特別不保事項
被保險人於本保險契約保險期間內，以乘客身分預定搭乘之定期航班 對於下列事項或該事項所致之損失，本公司不負理賠責任：
發生延誤，致被保險人實際出發時間較預定出發時間延誤四小時以上 一、因旅行社或公共交通工具業者破產、清算或債務不履行。
者，本公...

--- 相關條款 2 ---
來源: 南山 - 第二十六條 理賠文件
內容: [南山] 第二十六條 理賠文件
接班機實際出發之時止。
被保險人向本公司申請理賠時，應檢具下列文件：
下列情形視為同一事故，以給付一次為限：
一、共同文件：
一、去程於同一機場所發生之班機延誤或班機取消。
(一)理賠申請書。
二、回程於同一機場所發生之班機延誤或班機取消。
(二)費用單據正本。
三、...

--- 相關條款 3 ---
來源: 南山 - 第二十條 理賠文件 六、被保險人自行安排替代班機之目的地與原預定搭乘班機非屬同一
內容: [南山] 第二十條 理賠文件 六、被保險人自行安排替代班機之目的地與原預定搭乘班機非屬同一
被保險人向本公司申請理賠時，應檢具下列文件： 目的地。但僅變更轉機地而目的地未變更者，不在此限。
一、共同文件： 七、航空業者破產、清算或債務不履行。
(一)理賠申請書。
(二)旅行契約或交通工具之購票證明或...

--- 相關條款 4 ---
來源: 國泰 - 第二十一條 意外失能保險金、水陸大眾運輸交通意外失能保險金、航空大眾運輸交通意外失能保險金的申領
內容: [國泰] 第二十一條 意外失能保險金、水陸大眾運輸交通意外失能保險金、航空大眾運輸交通意外失能保險金的申領
受益人申領「意外失能保險金」、「水陸大眾運輸交通意外失能保險金」或「航空大眾運輸交通意外
失能保險金」時，應檢具下列文件：
一、保險金申請書。
二、保險單或其謄本。
三、失能診斷書；但必要時本...

--- 相關條款 5 ---
來源: 南山 - 第三條 理賠文件
內容: [南山] 第三條 

In [ ]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")

In [ ]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")


✅ AI 旅平險專員已上線！
💡 (輸入 'q' 即可結束)

👤 你的問題：買南山保險前氣象局已發布颱風警報，導致班機延誤會賠嗎？

=============== 提問確認 ===============
  買南山保險前氣象局已發布颱風警報，導致班機延誤會賠嗎？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：

根據【南山 - 第二十五條 特別不保事項】的規定：

「對於下列事項或該事項所致之損失，本公司不負理賠責任：

三、要保人向本公司申請訂立保險契約時，中華民國政府氣象機構已發布海上颱風警報。」

因此，如果中華民國政府氣象機構在您向南山產物申請訂立保險契約時已發布海上颱風警報，即使後續導致班機延誤，保險公司也不會負理賠責任。

----------------------------------------
📚 參考來源：
• 南山 : 第二十一條 承保範圍 第二十五條 特別不保事項
• 南山 : 第四十八條 理賠文件 主保險契約）後，加繳保險費，投保南山產物個人旅行不便保險班機
• 南山 : 第一條 理賠文件
• 南山 : 第五條 條款之適用
• 南山 : 第六條 保險期間的延長 第十六條 法令適用
• 南山 : 第一條 承保範圍
• 南山 : 第六條 海外突發疾病急診醫療保險金的給付
• 南山 : 第二十六條 理賠文件
----------------------------------------

👤 你的問題：q


In [ ]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")


✅ AI 旅平險專員已上線！
💡 (輸入 'q' 即可結束)

👤 你的問題：我和朋友去日本玩，晚餐吃海鮮結果兩個人都嚴重『食物中毒』送急診。請問這兩家保險公司會理賠嗎？

=============== 提問確認 ===============
  我和朋友去日本玩，晚餐吃海鮮結果兩個人都嚴重『食物中毒』送急診。請問這兩家保險公司會理賠嗎？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：

您好，根據您提供的【保險條款參考資料】，針對您和朋友在日本發生食物中毒送急診的情況，南山人壽的保險條款有相關規定：

1.  **關於「食物中毒」的認定與理賠金申領：**

    *   根據 **[南山] - 第十條 海外突發疾病醫療保險金的申領**
的規定：「食物中毒而引起者，即使只有一人，也視為『食物中毒』。」

    *   根據 **[南山] - 第三條 保險金的申領** 的規定，受益人申領「食物中毒保險金」時，應檢具下列文件：

        *   一、保險金申請書。

        *   二、保險單或其謄本。

        *   三、診斷證明書；但必要時本公司得要求提供二人以上發生食物中毒事故之證明文件，相關檢驗費用由本公司負擔。

        *   四、受益人之身分證明。

2.  **關於「海外突發疾病醫療保險金」的潛在適用性：**

    *   根據 **[南山] - 第二條 名詞定義**
的規定，「海外」係指臺灣、澎湖、金門、馬祖等由中華民國政府所轄範圍以外之地區。您在日本發生食物中毒符合「海外」的定義。

    *   根據 **[南山] - 第二條 名詞定義** 的規定，「突發疾病」係指被保險人需即時在醫院或診所診療始能避免損及身體健康之疾
病，且在本附加保險生效前一百八十日以內，未曾接受該疾病之診療者。若您的食物中毒情況符合此定義，且非屬於該條款列出的不負給付責任事項（例如美容
手術、健康檢查等），則可能符合「海外突發疾病」的理賠範圍。

    *   根據 **[南山] - 第十條 海外突發疾病醫療保險金的申領**
的規定，受益人申領本附加保險各項海外突發疾病醫療保險金時，應檢具下列文件：

        *   一、保險金申請書。

        *   二、保險單或其謄本。

        *   

In [ ]:
import os
import re
import pdfplumber
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import google.generativeai as genai

# ===  設定 API Key 與動態選擇模型 ===
API_KEY = "GOOGLE_API_KEY"
os.environ["GOOGLE_API_KEY"] = API_KEY
genai.configure(api_key=API_KEY)

available_models = [m.name.replace("models/", "") for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
chosen_model = next((m for m in available_models if "pro" in m or "flash" in m), available_models[-1] if available_models else None)

# ===  PDF 解析與清洗函數 ===
def process_insurance_pdf(file_path, company_name):
    full_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text: full_text += text + "\n"
    if company_name == "國泰":
        full_text = full_text.replace("樣張", "")
    full_text = re.sub(r'\n+', '\n', full_text)
    chunks = re.split(r'(第[一二三四五六七八九十百]+條\s+.*?\n)', full_text)

    return [
        Document(
            page_content=f"[{company_name}] {chunks[i].strip()}\n{chunks[i+1].strip() if (i+1) < len(chunks) else ''}",
            metadata={"保險公司": company_name, "條款標題": chunks[i].strip()}
        ) for i in range(1, len(chunks), 2)
    ]

# ===  重新建立唯一且乾淨的資料庫 ===
print("📚 正在重新解析 PDF 並建立乾淨的資料庫 (約需1分鐘)...")
documents = process_insurance_pdf("南山產物旅平險保單條款.pdf", "南山") + \
            process_insurance_pdf("國泰旅平險保單條款.pdf", "國泰")

embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")
# 注意這裡使用 from_documents 重新建立資料庫！
vector_db = Chroma.from_documents(documents=documents, embedding=embeddings)

# ===  初始化 RAG 提示詞 ===
llm = ChatGoogleGenerativeAI(model=chosen_model, temperature=0)
prompt_template = """
你是一位專業的旅平險 AI 專員。請根據以下提供的【保險條款參考資料】來回答客戶的問題。
【嚴格要求】
1. 你的回答必須完全基於參考資料，絕不能自己瞎編。
2. 回答時，必須明確標註來源！例如：「根據 [保險公司] - [條款標題] 的規定...」。
3. 如果參考資料中沒有答案，請誠實回答「根據目前條款無法得知」。

【保險條款參考資料】
{context}

客戶問題：{question}
專業回答：
"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# ===  啟動對話迴圈 (參數已調整為 k=8 解決單一公司問題) ===
print("\n✅ AI 旅平險專員已重新上線！(資料庫已更新完畢)")
print("💡 (在下方輸入框打字，輸入 'q' 即可結束對話)\n")

while True:
    user_input = input("👤 你的問題：")
    if user_input.lower() in ['q', '退出', 'quit', 'exit']:
        print("\n👋 結束對話，AI 專員下線囉！")
        break
    if not user_input.strip(): continue

    print("\n⏳ 系統檢索與思考中...\n")
    retrieved_docs = vector_db.similarity_search(user_input, k=8)

    # 過濾重複抓取的文獻，確保不會印出雙胞胎
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}]\n來源: {doc.metadata['保險公司']} - {doc.metadata['條款標題']}\n內容: {doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    response = llm.invoke(PROMPT.format(context=context_text, question=user_input))

    print("🤖 AI 專員回答：")
    print(response.content)
    print("\n--------------------------------------------------")
    print("📚 本次回答參考的唯一底層文獻：")
    for doc in unique_docs:
        print(f"- {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print("==================================================\n")

In [ ]:
import os
import re
import pdfplumber
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import google.generativeai as genai

# ===  設定 API Key 與動態選擇模型 ===
API_KEY = "GOOGLE_API_KEY"
os.environ["GOOGLE_API_KEY"] = API_KEY
genai.configure(api_key=API_KEY)

available_models = [m.name.replace("models/", "") for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
chosen_model = next((m for m in available_models if "pro" in m or "flash" in m), available_models[-1] if available_models else None)

# ===  PDF 解析與清洗函數 ===
def process_insurance_pdf(file_path, company_name):
    full_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text: full_text += text + "\n"
    if company_name == "國泰":
        full_text = full_text.replace("樣張", "")
    full_text = re.sub(r'\n+', '\n', full_text)
    chunks = re.split(r'(第[一二三四五六七八九十百]+條\s+.*?\n)', full_text)

    return [
        Document(
            page_content=f"[{company_name}] {chunks[i].strip()}\n{chunks[i+1].strip() if (i+1) < len(chunks) else ''}",
            metadata={"保險公司": company_name, "條款標題": chunks[i].strip()}
        ) for i in range(1, len(chunks), 2)
    ]

# ===  重新建立唯一且乾淨的資料庫 ===
print("📚 正在重新解析 PDF 並建立乾淨的資料庫 (約需1分鐘)...")
documents = process_insurance_pdf("南山產物旅平險保單條款.pdf", "南山") + \
            process_insurance_pdf("國泰旅平險保單條款.pdf", "國泰")

embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")
# 注意這裡使用 from_documents 重新建立資料庫！
vector_db = Chroma.from_documents(documents=documents, embedding=embeddings)

# ===  初始化 RAG 提示詞 ===
llm = ChatGoogleGenerativeAI(model=chosen_model, temperature=0)
prompt_template = """
你是一位專業的旅平險 AI 專員。請根據以下提供的【保險條款參考資料】來回答客戶的問題。
【嚴格要求】
1. 你的回答必須完全基於參考資料，絕不能自己瞎編。
2. 回答時，必須明確標註來源！例如：「根據 [保險公司] - [條款標題] 的規定...」。
3. 如果參考資料中沒有答案，請誠實回答「根據目前條款無法得知」。

【保險條款參考資料】
{context}

客戶問題：{question}
專業回答：
"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# ===  啟動對話迴圈 (參數已調整為 k=8 解決單一公司問題) ===
print("\n✅ AI 旅平險專員已重新上線！(資料庫已更新完畢)")
print("💡 (在下方輸入框打字，輸入 'q' 即可結束對話)\n")

while True:
    user_input = input("👤 你的問題：")
    if user_input.lower() in ['q', '退出', 'quit', 'exit']:
        print("\n👋 結束對話，AI 專員下線囉！")
        break
    if not user_input.strip(): continue

    print("\n⏳ 系統檢索與思考中...\n")
    retrieved_docs = vector_db.similarity_search(user_input, k=8)

    # 過濾重複抓取的文獻，確保不會印出雙胞胎
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}]\n來源: {doc.metadata['保險公司']} - {doc.metadata['條款標題']}\n內容: {doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    response = llm.invoke(PROMPT.format(context=context_text, question=user_input))

    print("🤖 AI 專員回答：")
    print(response.content)
    print("\n--------------------------------------------------")
    print("📚 本次回答參考的唯一底層文獻：")
    for doc in unique_docs:
        print(f"- {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print("==================================================\n")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


📚 正在重新解析 PDF 並建立乾淨的資料庫 (約需1分鐘)...


modules.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]


✅ AI 旅平險專員已重新上線！(資料庫已更新完畢)
💡 (在下方輸入框打字，輸入 'q' 即可結束對話)

👤 你的問題：旅遊途中遇到飯店火災，造成全身 25% 面積二度燒燙傷，但幸運地沒有造成殘廢失能。請問除了實支實付的醫藥費外，這兩家會額外給付『一筆定額的保險金』嗎？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：
您好！針對您的問題，我將根據提供的保險條款參考資料進行說明：

**關於國泰人壽：**

是的，根據國泰的條款，除了實支實付的醫藥費外，可能會額外給付一筆定額的「重大燒燙傷保險金」。

根據 [國泰] - 第三條 保險範圍 的規定，被保險人於契約有效期間內，因遭受意外傷害事故，致其身體蒙受傷害而致成身故、失能或重大燒燙傷時，本公司依照契約約定給付保險金。

您的情況「旅遊途中遇到飯店火災，造成全身 25% 面積二度燒燙傷」符合 [國泰] - 第十二條 重大燒燙傷保險金的給付 中所列的條件：
*   被保險人遭受意外傷害事故而蒙受燒燙傷之傷害。
*   燒燙傷面積達全身百分之二十以上（您的情況為 25%）。

若符合上述條件，根據 [國泰] - 第十二條 重大燒燙傷保險金的給付 的規定，國泰人壽將按本契約保險單上所記載之保險金額的**百分之二十**給付「重大燒燙傷保險金」，且此給付以一次為限。

**關於南山人壽：**

根據目前提供的 [南山] - 第二條 定義 一、個人海外旅行不便保險、[南山] - 第二條 承保範圍、[南山] - 第二條 名詞定義 五、被保險人非法施用防制毒品相關法令所稱之毒品 等條款參考資料，**無法得知**南山人壽是否有提供針對燒燙傷的額外定額保險金給付。這些條款主要涵蓋旅行不便、緊急救援、失能或死亡保險金，以及海外突發疾病的定義與除外責任，並未提及重大燒燙傷的定額給付項目。

--------------------------------------------------
📚 本次回答參考的唯一底層文獻：
- 國泰 : 第二十二條 重大燒燙傷保險金的申領
- 國泰 : 第十二條 重大燒燙傷保險金的給付
- 南山 : 第二條 定義 一、個人海外旅行不便保險：
- 國泰 : 第三條 保險範圍
- 國泰 : 第二條 傷害醫療保險金的給付
- 南山 : 第二條 承保範圍
- 國泰 : 第二十一條 意外失能保險金、水陸大

In [ ]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")


✅ AI 旅平險專員已上線！
💡 (輸入 'q' 即可結束)

👤 你的問題：旅遊途中遇到飯店火災，造成全身 25% 面積二度燒燙傷，但幸運地沒有造成殘廢失能。請問除了實支實付的醫藥費外，這兩家會額外給付『一筆定額的保險金』嗎？

=============== 提問確認 ===============
  旅遊途中遇到飯店火災，造成全身 25%
  面積二度燒燙傷，但幸運地沒有造成殘廢失能。請問除了實支實付的醫藥費外，這兩家會額外給付『一筆定額的保險金』嗎？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：

您好，我是您的旅平險 AI 專員。針對您提出的問題，我將根據提供的條款資料進行說明：

**關於國泰人壽 (Cathay Life) 的部分：**

根據您描述的「旅遊途中遇到飯店火災，造成全身 25% 面積二度燒燙傷，但幸運地沒有造成殘廢失能」的情況，國泰人壽會額外給付一筆定額的保險金。

1.  **保險範圍確認：**

    根據 **[國泰] - 第三條 保險範圍** 的規定，被保險人於保險契約有效期間內，因遭受意外傷害事故，致其身體蒙受傷害而致成身故、
失能或重大燒燙傷時，保險公司會依照契約約定給付保險金。飯店火災造成的燒燙傷屬於意外傷害事故。

2.  **重大燒燙傷保險金給付條件：**

    根據 **[國泰] - 第十二條 重大燒燙傷保險金的給付** 的規定，被保險人遭受意外傷害事故而蒙受燒燙傷之傷害，並自意外傷害事故發
生之日起一百八十日以內，經醫師診斷符合「燒燙傷面積達全身百分之二十以上」者，保險公司會按本契約保險單上所記載之保險金額的百分之二十給付「重大
燒燙傷保險金」。

    您的情況是「全身 25% 面積二度燒燙傷」，這符合條款中「燒燙傷面積達全身百分之二十以上」的條件。

3.  **給付金額與次數：**

    因此，國泰人壽會按您保險單上所記載之保險金額的百分之二十給付「重大燒燙傷保險金」。此項保險金的給付以一次為限。

4.  **申領文件：**

    根據 **[國泰] - 第二十二條 重大燒燙傷保險金的申領** 的規定，申領「重大燒燙傷保險金」時應檢具保險金申請書、保險單或其謄本
、醫療診斷書或住院證明（需載明燒燙傷程度及佔體表面積之比例），以及受益人之身分證明。

**關於南山人壽 (Nansha

KeyboardInterrupt: Interrupted by user